In [1]:
import pandas as pd

## **SET UP DATA FOR SENTIMENT TRAINING**

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [7]:
global_review = pd.read_parquet("../data/review_global.parquet")

sentiment_df = global_review[global_review["stars"] != 3].copy()

sentiment_df["sentiment"] = sentiment_df['stars'].apply(lambda x: 1 if x > 3 else 0).astype(int)


x = sentiment_df['text']
y = sentiment_df['sentiment']

x_train_tmp, x_test, y_train_tmp, y_test = train_test_split(x,y, test_size=0.2, random_state= 42, stratify= y)
x_train_raw, x_val, y_train_raw, y_test_val = train_test_split(x_train_tmp,y_train_tmp, test_size=0.2, random_state= 42, stratify= y_train_tmp)

# Class imbalance fix (split up positive reviews vs negative reviews)
df_raw = pd.DataFrame({'text': x_train_raw, 'sentiment': y_train_raw})  # patch the x y training data to sample

pos_review = df_raw[df_raw["sentiment"] == 1]  
neg_review = df_raw[df_raw["sentiment"] == 0] 

print(len(pos_review))
print(len(neg_review))

pos_sample = pos_review.sample(n=len(neg_review), random_state= 42)     # sample positive reviews - match with number of negative reviews

df = pd.concat([pos_sample,neg_review]).sample(frac = 1, random_state= 42).reset_index(drop=True)   # patch positive reviews (matched) and negative reviews together

print(df['sentiment'].value_counts())

x_train = df['text']
y_train = df['sentiment']

print('DONE!')


2054294
621902
sentiment
1    621902
0    621902
Name: count, dtype: int64
DONE!


## **TF-IDF (Term Frequency-Inverse Document Frequency)**

TF-IDF is a statistical method used in natural language processing and information retrieval to evaluate how important a word is to a document in relation to a larger collection of documents. TF-IDF combines two components:
$$TF(t,d) = \frac{\text{Number of times term t appears in document d}}{\text{Total numer of terms in document d}}$$
$$IDF(t,D) = \log \frac{\text{Total number of documents in corpus D"}}{\text{"Number of documents containing term t}}$$
$$TD-IDF(t,d,D) = TF(t,d) \times IDF(t,D)$$

documentation link: 
- https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/

Initializing TF-IDF through scikit-learn:
1. `max_feature`: used to build a vocabulary that only consider the top max_features ordered by term frequency across the corpus. Otherwise, all features are used (default). For the model, I initiate around 10000 features (vocabulary)
2. `stop_words`: To automatically remove words like "is", "at", "which" in a string. It is passed to _check_stop_list and the appropriate stop list is returned. ‘english’ is currently the only supported string value.
3. `ngram_range`: set to (1,2). I basically tell the Vectorizer to score both individual words (unigrams) and pairs of adjacent words (bigrams)

## **Simple Linear Model (LOGISTIC REGRESSION)**

We will use the vectorized training data to feed a binary classifier (logistic Regression) to recoginize the sentiment pattern of the reviews

In [9]:
tfidf = TfidfVectorizer(max_features=10000, stop_words="english", ngram_range=(1,2))    # Initialize tf-idf

x_train_vec = tfidf.fit_transform(x_train)

x_val_vec = tfidf.transform(x_val)
x_test_vec = tfidf.transform(x_test)

model = LogisticRegression(max_iter= 1000, random_state= 42)
model.fit(x_train_vec, y_train)

y_val_pred = model.predict(x_val_vec)

print(classification_report(y_test_val, y_val_pred)) # Named the variable wrong, my bad

              precision    recall  f1-score   support

           0       0.87      0.96      0.91    155476
           1       0.99      0.96      0.97    513574

    accuracy                           0.96    669050
   macro avg       0.93      0.96      0.94    669050
weighted avg       0.96      0.96      0.96    669050



The classification report return an exceptionally impressive results:
- 0.87 precision and 0.96 recall for negative reviews
- 0.99 precision and 0.96 recall for positive reviews

This result would make the simple Logistic Regression model strong and reliable enough to take on sentimental classification tasks. I will also try a fine-tuned Transformer mdoel to evaluate the tradeoffs between a simple Logistic Regression with TF-IDF vectorizer and a deep-learning model

---

### **Save the LR Model**

In [10]:
import joblib

joblib.dump(model, "sentiment_logistic_model.joblib")
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')

print("model and vectorizer saved!!")

model and vectorizer saved!!
